# 02.6 阶段项目：数字分类对比实验

这一份 notebook 是 `Phase 2` 的综合项目。  

目标不是只把一个模型跑通，而是做一个小型实验对比：  

- MLP 基线模型（MLP baseline）
- CNN 模型（CNN model）
- 对比验证集和测试集表现
- 对最优模型做错误分析

## 学习目标

学完后你应该能

1. 搭建一个小型视觉实验项目
2. 对比 MLP 和 CNN 在图像任务上的差异
3. 用统一训练函数管理不同模型
4. 整理对比结果表
5. 对最优模型做测试评估和简单错误分析
6. 更像做项目一样思考，而不只是做单题练习

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


## 1. 准备数据

这里继续使用 `digits`，因为它足够小，适合做模型对比实验。  


In [ ]:
digits = load_digits()
images = digits.images
labels = digits.target

X_train_full, X_test, y_train_full, y_test = train_test_split(
    images,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full,
)

print("train:", X_train.shape)
print("val:", X_val.shape)
print("test:", X_test.shape)

In [ ]:
train_mean = float(X_train.mean() / 16.0)
train_std = float(X_train.std() / 16.0)

transform = transforms.Compose([
    transforms.Lambda(lambda x: x.float() / 16.0),
    transforms.Normalize(mean=[train_mean], std=[train_std]),
])

In [ ]:
class DigitsProjectDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = torch.tensor(self.images[index], dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(int(self.labels[index]), dtype=torch.long)
        if self.transform is not None:
            image = self.transform(image)
        return image, label


train_ds = DigitsProjectDataset(X_train, y_train, transform=transform)
val_ds = DigitsProjectDataset(X_val, y_val, transform=transform)
test_ds = DigitsProjectDataset(X_test, y_test, transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

xb, yb = next(iter(train_loader))
print("xb.shape =", xb.shape)
print("yb.shape =", yb.shape)

## 2. 定义两个模型

这里的目标是让两个模型共享同一套数据输入，只在结构上不同。  


In [ ]:
class MLPBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 2 * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


print(MLPBaseline())
print()
print(CNNModel())

## 3. 通用训练函数

为了公平对比，两个模型尽量共用同一套训练逻辑。  


In [ ]:
def batch_accuracy(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_acc += batch_accuracy(logits, yb)
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches


def train_model(model, train_loader, val_loader, epochs=5, lr=0.01):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
            }
        )

    return pd.DataFrame(history), model

## 4. 训练 MLP Baseline

MLP 可以作为一个很好的 baseline，因为它更简单、更快。  


In [ ]:
torch.manual_seed(0)
mlp_history, mlp_model = train_model(MLPBaseline(), train_loader, val_loader, epochs=5, lr=0.01)
print(mlp_history)

## 5. 训练 CNN

CNN 会显式利用图像的空间结构，这是它相对 MLP 的关键优势。  


In [ ]:
torch.manual_seed(0)
cnn_history, cnn_model = train_model(CNNModel(), train_loader, val_loader, epochs=5, lr=0.01)
print(cnn_history)

## 6. 对比验证结果

这里把两种模型的最终验证结果放到一起。  


In [ ]:
comparison = pd.DataFrame(
    {
        "model": ["MLPBaseline", "CNNModel"],
        "final_train_acc": [mlp_history.iloc[-1]["train_acc"], cnn_history.iloc[-1]["train_acc"]],
        "final_val_acc": [mlp_history.iloc[-1]["val_acc"], cnn_history.iloc[-1]["val_acc"]],
        "final_val_loss": [mlp_history.iloc[-1]["val_loss"], cnn_history.iloc[-1]["val_loss"]],
    }
)

print(comparison)

## 7. 在测试集上评估最优模型

这里简单按最终验证准确率选更优模型。  


In [ ]:
best_model_name = comparison.sort_values("final_val_acc", ascending=False).iloc[0]["model"]
best_model = mlp_model if best_model_name == "MLPBaseline" else cnn_model

print("best model =", best_model_name)

In [ ]:
best_model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for xb, yb in test_loader:
        logits = best_model(xb)
        preds = logits.argmax(dim=1)
        all_preds.append(preds)
        all_targets.append(yb)

all_preds = torch.cat(all_preds)
all_targets = torch.cat(all_targets)

test_acc = accuracy_score(all_targets.numpy(), all_preds.numpy())
cm = confusion_matrix(all_targets.numpy(), all_preds.numpy())

print("test accuracy =", test_acc)
print("confusion matrix =\n", cm)

## 8. 简单错误分析

除了一个总准确率数字，我们还想看看模型错在了哪里。  


In [ ]:
mis_idx = (all_preds != all_targets).nonzero(as_tuple=False).squeeze(1)
print("number of mistakes =", len(mis_idx))

In [ ]:
num_show = min(6, len(mis_idx))

if num_show > 0:
    plt.figure(figsize=(9, 4))
    for i in range(num_show):
        idx = mis_idx[i].item()
        image = X_test[idx]
        true_label = y_test[idx]
        pred_label = all_preds[idx].item()

        plt.subplot(2, 3, i + 1)
        plt.imshow(image, cmap="gray")
        plt.title(f"true={true_label}, pred={pred_label}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No mistakes to display.")

## 9. 对新样本做推理

这里拿测试集前几张图像做一个简单推理展示。  


In [ ]:
sample_images = []
sample_labels = []

for i in range(6):
    image, label = test_ds[i]
    sample_images.append(image)
    sample_labels.append(label.item())

sample_batch = torch.stack(sample_images)

with torch.no_grad():
    logits = best_model(sample_batch)
    preds = logits.argmax(dim=1).tolist()

plt.figure(figsize=(9, 4))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(sample_images[i].squeeze(0), cmap="gray")
    plt.title(f"true={sample_labels[i]}, pred={preds[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 练习 1
# 把 MLP 的隐藏层从 128 改成 256，再重新比较结果。
# Change the MLP hidden size from 128 to 256 and compare the results again.

In [ ]:
# 练习 2
# 给 CNN 加一个 Dropout 层，再观察验证集准确率是否变化。
# Add a Dropout layer to the CNN and observe whether validation accuracy changes.

In [ ]:
# 练习 3
# 用一句话解释为什么 CNN 往往比 MLP 更适合图像任务。
# In one sentence, explain why CNNs are often better suited than MLPs for image tasks.

参考回答

因为 CNN 能显式利用局部邻域和空间结构，而 MLP 展平后会丢失这些图像结构信息。  


## 10. 小结

这个项目的重点不是“跑通一个模型”，而是“做一次有对比、有结论的小实验”。  

你现在已经练到的内容

- 统一数据输入（unified data pipeline）
- 多模型对比（multi-model comparison）
- 训练结果表（result tables）
- 测试评估（test evaluation）
- 错误分析（error analysis）

这已经非常接近真实的小型深度学习项目工作流。  
